# 01 – BM25-Index aufbauen (PyTerrier)

Dieses Notebook baut den **Pyterrier-BM25-Index** aus dem LongEval-SCI-Korpus (Snapshot 3).
Der Index wird lokal gespeichert und von allen nachfolgenden Retrieval-Schritten wiederverwendet.

> **Wichtig:** Cell 4 (Index bauen) muss **nur einmal ausgeführt** werden. Danach kann der Index
> direkt über `pt.IndexFactory.of(index_path)` geladen werden.

**Voraussetzung:** `00_Data_loading.ipynb` wurde erfolgreich ausgeführt.

**Ausgabe:** `Index_Longeval_SCI_snapshot3/` (binärer Terrier-Index, ~einige GB)

## 1. PyTerrier initialisieren

Startet die Java-VM, die Terrier intern benötigt.

In [7]:
import pyterrier as pt

# PyTerrier starten (initialisiert die Java-VM)
if not pt.started():
    pt.init()

C:\Users\lenna\AppData\Local\Temp\ipykernel_32772\1222611476.py:4: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


## 2. Dokumentkorpus einlesen

Liest die JSONL-Dokumentdateien aus dem lokalen ir-datasets-Cache zeilenweise ein.
Titel und Abstract werden zu einem einzigen `text`-Feld zusammengeführt,
das dann indiziert wird.

In [8]:
# Dokumentkorpus laden – Pfad zum lokalen ir-datasets-Cache
import json
from pathlib import Path

docs_path = (
    Path.home() / ".ir_datasets" / "longeval-sci-2026"
    / "longeval_sci_09_11_2026_documents" / "data" / "processed"
    / "doc_collection_09032026_abstract_2" / "snapshot-3"
    / "longeval_sci_test-09-11_2026_abstract" / "documents"
)

def document_generator():
    """Generator: liest alle JSONL-Dateien und liefert Dicts mit docno und text."""
    for file in sorted(docs_path.glob("documents_*.jsonl")):
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                yield {
                    "docno": doc["id"],
                    # Titel + Abstract als gemeinsames Textfeld (für BM25-Indizierung)
                    "text": f"{doc.get('title', '')} {doc.get('abstract', '')}".strip()
                }

## 3. Index-Pfad festlegen

Der Index wird relativ zum aktuellen Arbeitsverzeichnis gespeichert.

In [9]:
# Index-Pfad relativ zum Notebook-Verzeichnis definieren
BASE_DIR = Path.cwd()
index_path = str(BASE_DIR / "Index_Longeval_SCI_snapshot3")

## 4. Index bauen

> **Nur einmal ausführen!** Der Prozess dauert je nach Hardware 20–60 Minuten
> und indiziert ~1,66 Mio. Dokumente. Danach diese Cell auskommentieren.

Die `meta`-Einstellung speichert `docno` und den vollen `text` direkt im Index,
damit Retrieval-Ergebnisse ohne Nachladen dekodiert werden können.

In [ ]:
# # Index aufbauen – NUR EINMAL AUSFÜHREN, danach auskommentieren!
# from pyterrier import IterDictIndexer

# indexer = IterDictIndexer(
#     index_path=index_path,
#     overwrite=True,                    # Überschreibt bestehenden Index
#     meta={"docno": 100, "text": 20480} # Metadaten werden direkt im Index gespeichert
# )

# indexer.index(document_generator())

## 5. Index laden und Statistiken prüfen

Lädt den fertigen Index und gibt Kennzahlen aus.
Erwartete Werte: ~1,66 Mio. Dokumente, ~2,26 Mio. Terme, ~281 Mio. Tokens.

In [10]:
# Index laden und Statistiken prüfen
index = pt.IndexFactory.of(index_path)
print(index.getCollectionStatistics().toString())

16:54:56.041 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1,3 GiB of memory would be required.
Number of documents: 1661900
Number of terms: 2255473
Number of postings: 169210872
Number of fields: 0
Number of tokens: 281960829
Field names: []
Positions:   false



## 6. Sanity-Check: BM25-Testabfrage

Führt eine einfache Testabfrage aus, um sicherzustellen, dass
der Index korrekt funktioniert und Ergebnisse liefert.

In [11]:
# Schneller Funktionstest: BM25-Retrieval auf dem fertigen Index
bm25 = pt.terrier.Retriever(index, wmodel="BM25", metadata=["docno", "text"])
results = bm25.search("peer-to-peer communication")
results.head()

,qid,docid,docno,text,rank,score,query
0,1,1183054,303096251,Komunikasi Efektif Kepala Madrasah Ibtidaiyah ...,0,6.794784,peer-to-peer communication
1,1,86795,6922555,2006 Nebraska Rural Poll Results: Views of Lif...,1,6.788197,peer-to-peer communication
2,1,1604682,312632655,КОМУНІКАТИВНА КРЕАТИВНІСТЬ У СТРУКТУРІ ПРОФЕС...,2,6.777475,peer-to-peer communication
3,1,325145,75782819,A Cross-Lagged Panel Analysis of the Relations...,3,6.777321,peer-to-peer communication
4,1,1042719,301425028,Formulating the Novelty of Communication Resea...,4,6.776863,peer-to-peer communication


In [ ]:
import pandas as pd

queries_path = Path.cwd().parent / ".ir_datasets" / "longeval-sci-2026" / "longeval_adhoc-queries-snapshot-test.tsv"

df = pd.read_csv(queries_path, sep="\t")
print(df.shape)
print(df.head())
print(df.columns.tolist())